In [3]:
from azure.ai.ml import MLClient
from azure.ai.ml.entities import Model
from azure.ai.ml.constants import AssetTypes
from azure.identity import DefaultAzureCredential

In [5]:
# 1. Cargar métricas e hiperparámetros guardados
import json

with open('../outputs/metrics.json') as f:
    metrics = json.load(f)

with open ('../outputs/hyperparams.json') as f:
    hyperparams = json.load(f)

In [6]:
# 2. Conectar a ws
ml_client = MLClient.from_config(credential=DefaultAzureCredential())

Found the config file in: /config.json
Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


In [7]:
print(ml_client.workspace_name)

mlw-churn-dev


In [14]:
# 3. Registrar modelo
model = Model(
    path='../outputs/model.pkl',
    type=AssetTypes.CUSTOM_MODEL,
    name='telco-churn-model',
    # version=2,
    description='Telco customer churn prediction - HistGradientBoosting',
    tags={
        # Framework
        'framework': 'scikit-learn',
        'algorithm': 'HistGradientBoosting',

        # Metricas
        'accuracy': f"{metrics['accuracy']:.4f}",
        'precision': f"{metrics['precision']:.4f}",
        'recall': f"{metrics['recall']:.4f}",
        'f1_score': f"{metrics['f1_score']:.4f}",

        # Hiperparametros clave
        'learning_rate': str(hyperparams['learning_rate']),
        'max_depth': str(hyperparams['max_depth']),
        'max_iter': str(hyperparams['max_iter']),
        'class_weight': 'balanced',

        # Configuración
        'threshold': '0.3', 

        #
        'stage': 'candidate',
    }
)

registered_model = ml_client.models.create_or_update(model)

print(f"Modelo registrado: {registered_model.name}, versión: {registered_model.version}")

Uploading model.pkl (< 1 MB): 100%|██████████| 725k/725k [00:00<00:00, 15.1MB/s]




Modelo registrado: telco-churn-model, versión: 3


In [16]:
registered_model.tags['threshold'] = 'pending_analysis'
ml_client.models.create_or_update(registered_model)

Model({'job_name': None, 'intellectual_property': None, 'system_metadata': None, 'is_anonymous': False, 'auto_increment_version': False, 'auto_delete_setting': None, 'name': 'telco-churn-model', 'description': 'Telco customer churn prediction - HistGradientBoosting', 'tags': {'framework': 'scikit-learn', 'algorithm': 'HistGradientBoosting', 'accuracy': '0.7271', 'precision': '0.4917', 'recall': '0.7968', 'f1_score': '0.6082', 'learning_rate': '0.05', 'max_depth': '3', 'max_iter': '200', 'class_weight': 'balanced', 'threshold': 'pending_analysis', 'stage': 'candidate'}, 'properties': {}, 'print_as_yaml': False, 'id': '/subscriptions/9cbbe496-fe29-459a-a3d7-44790ebac1fb/resourceGroups/rg-churn-dev/providers/Microsoft.MachineLearningServices/workspaces/mlw-churn-dev/models/telco-churn-model/versions/3', 'Resource__source_path': '', 'base_path': '/mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-churn-dev/code/Users/asandovh/telco-churn/notebooks', 'creation_context': <azure.ai.ml.entitie